# JetRacer - Tune bam vach truc tiep tren JupyterLab

Notebook nay thay cho viec SSH vao Jetson go lenh. Mo tu may khac, keo slider, nhin ngay ket qua tren dung frame camera dang chay, bam mot nut la xe tu bam line.

**Truoc moi lan Run All**: chon `Kernel > Restart & Clear Output` de kernel cu release camera. Hai notebook mo camera cung luc se bao `Failed to create CaptureSession`.

**AN TOAN**

- Mo len la o trang thai DUNG. Khong co lenh nao xuong phan cung cho den khi bam **2. CHAY - BAM LINE**.
- Ga tang dan trong 1 giay dau, khong nhay thang len muc chay.
- Giao dien tu tu choi CHAY neu dang mat vach tren 20% frame.
- **DUNG KHAN CAP** cat ga ngay. Camera dung hoac loi xu ly cung tu dong cat ga.
- Lan CHAY dau tien: **ke banh khoi mat dat** de kiem chieu danh lai truoc.

## 1. Kiem tra thu muc va file

In [ ]:
%cd /home/jetson/JetsonRacer

import os
import socket

required = [
    'tools/tune_lane_jupyter.py',
    'src/jetracer_baseline/tuning_ui.py',
    'src/jetracer_baseline/perception/lane.py',
    'src/jetracer_baseline/perception/shading.py',
    'configs/default.yaml',
]
missing = [p for p in required if not os.path.exists(p)]
print('Hostname:', socket.gethostname())
print('Thu muc:', os.getcwd())
print('Files:', 'OK' if not missing else 'THIEU ' + ', '.join(missing))
if missing:
    raise IOError('Chua copy du code moi sang Jetson')

print('Shading da hieu chuan:', os.path.exists('configs/shading.yaml'))

## 2. Mo giao dien

Mo len xe o trang thai **DUNG** - khong co lenh nao xuong phan cung. Chi khi bam **2. CHAY - BAM LINE** xe moi tu chay.

Thu tu thao tac:

1. Bam **1. MO CAMERA**, doi preview hien ra.
2. Chon **Mau vach** dung sa ban: `do` cho sa ban tap, `trang` cho sa ban thi.
3. Nhin o **MASK (nguong mau)** goc tren phai. Muc tieu: chi con vach, khong con dom nhieu, khong dinh vien lane. Keo slider tab **Bam vach** cho den khi mask sach VA `dai` trong banner >= 4.
4. **KE BANH KHOI MAT DAT**, bam **CHAY**, xoay xe cho vach sang ben TRAI: `cte` phai AM va banh phai quay TRAI. Neu banh quay nguoc -> bam **DUNG KHAN CAP**, doi dau `control.driver.steering_gain` trong `configs/default.yaml`.
5. Dat xe xuong dung tren vach, bam **CHAY**. Ga tang dan trong 1 giay dau.
6. Bam **LUU CONFIG** khi da vua y.

In [ ]:
%run tools/tune_lane_jupyter.py

## 3. Chan doan nhanh

| Nhin thay | Keo slider nao |
|---|---|
| Mask day dom trang | Tang `Bao hoa toi thieu (S)` hoac `Dien tich blob toi thieu` |
| Mask trong, banner bao MAT VACH | Giam `Bao hoa toi thieu (S)` va `Dien tich blob toi thieu` |
| Duong do bam vao vien lane | Giam `Be rong cum toi da` |
| `dai` chi 1-2 | Giam `Dien tich blob toi thieu` va `Pixel toi thieu / dai` |
| Nen van phong lot vao mask | Tang `ROI tren` |
| Duong xanh rang cua (dao dong) | Giam `PID Kp`, giam `Trong so diem ngam` |
| Xe cat cua muon | Tang `Trong so diem ngam` |
| Vao cua vang ra | Tang `Bo ga theo cua` |
| Ga luon o muc thap nhat | Giam `Bo ga theo cua` |

Bang so lieu duoi preview tu canh bao khi vuot muc tieu (mat vach > 2%, cte_rms > 0.15, lai doi dau > 3 lan/giay).

Giao dien tu tu choi CHAY neu dang mat vach tren 20% frame - luc do xe se lao theo gia tri `cte` cu.

## 4. Xem thu ma chac chan banh khong quay

Chi can khi muon dat xe tren ban ma van yen tam bam CHAY (vi du dang demo).

Binh thuong KHONG can chay cell nay - cell muc 2 da du, va no cung khong cho xe chay cho den khi ban bam nut CHAY.

In [ ]:
try:
    ui.close()
except NameError:
    pass

from tools.tune_lane_jupyter import launch
ui = launch(driver_kind='dryrun')

## 5. Chay luot day du bang config vua luu

Giao dien tune KHONG ghi log CSV. So lieu chinh thuc phai lay tu mot luot chay day du qua CLI - do moi la con so dua vao Technical Paper.

In [ ]:
!python3 -m src.jetracer_baseline.cli run --task speed --driver nvidia \n    --override configs/tuned.yaml --max-seconds 60 --record

## 6. Dong truoc khi tat notebook

In [ ]:
ui.close()